This notebook collects all CDCR-specific data for state facilities, for exposure and vulnerability indices. The data produced gets joined to climate hazards in a different file.

In [1]:
import re

import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
# Load all CA facilities and filter to CDCR state prisons (type == 'STATE')
# ca_facilities.csv is the output of create_facilities.ipynb in ca_prison_climate_justice (sibling checkout)
facilities = pd.read_csv("../ca_prison_climate_justice/data_sources/facilities/ca_facilities.csv")
facilities = facilities[facilities['type'] == 'STATE'].reset_index(drop=True)
len(facilities)

84

## Import data

In [3]:
# 2025 average in-custody population per institution: the mean of the monthly
# counts scraped from the CDCR Population Data Points dashboard
# (scrapers/fetch_cdcr_population.js).
pop_monthly = pd.read_csv("data/cdcr_population_by_location.csv")
cdcr_pop = (
    pop_monthly[pop_monthly['year'] == 2025]
    .groupby('cdcr_code')['in_custody'].mean().round(0)
    .reset_index()
    .rename(columns={'cdcr_code': 'Facility_Code', 'in_custody': 'Average_2025_Population'})
)
len(cdcr_pop)

37

In [4]:
cdcr_age = pd.read_csv("sources/cdcr_in-custody-age_2025.csv")
cdcr_countryofbirth = pd.read_csv("sources/cdcr_in-custody-countryofbirth_2025.csv")
cdcr_gender = pd.read_csv("sources/cdcr_in-custody-gender_2025.csv")
cdcr_race = pd.read_csv("sources/cdcr_in-custody-race_2025.csv")

In [5]:
cdcr_metadata = pd.read_csv("sources/cdcr_manual_data.csv")
len(cdcr_metadata)

33

## Get CDCR facility abbreviation code

In [6]:
def process_cdcr_name(df):
    """
    Processes the facility name to get CDCR facility attributes.

    - Extract CDCR codes from names (e.g., (KVSP)). Manually set Avenal State Prison (10000826 -> ASP).
    - Correct FEMA acronym mismatches: CCFW -> CCWF (letter transposition), FSP -> FOL (naming convention).
    - Identify fire camps.
    """
    
    df['CDCR_code'] = None
    df['CDCR_firecamp'] = False

    # FEMA name parentheticals that don't match CDCR's own codes
    fema_to_cdcr = {
        'CCFW': 'CCWF',  # Central California Women's Facility: FEMA has letters transposed
        'FSP': 'FOL',    # Folsom State Prison: FEMA uses FSP, CDCR uses FOL
    }

    def extract_info(row):
            
        name = str(row['name'])
        
        # Check for Fire Camp (Case insensitive search for 'CAMP')
        if 'CAMP' in name.upper():
            row['CDCR_firecamp'] = True
            
        # Extract Code in parenthesis: e.g., "Kern Valley (kvsp)"
        code_match = re.search(r'\(([^)]+)\)', name)
        if code_match:
            row['CDCR_code'] = code_match.group(1).strip().upper()
            # Remove the parenthesis and its content from the name
            name = re.sub(r'\s*\([^)]+\)', '', name).strip()

        # Correct FEMA acronyms that don't match CDCR's codes
        if row['CDCR_code'] in fema_to_cdcr:
            row['CDCR_code'] = fema_to_cdcr[row['CDCR_code']]

        # Manual override for Avenal State Prison (no parenthetical in FEMA name)
        if row['facilityid'] == 10000826:
            row['CDCR_code'] = 'ASP'
        
        row['name'] = name
        return row

    # Apply the extraction and cleaning logic row-wise
    df = df.apply(extract_info, axis=1)
    
    return df

In [7]:
facilities = process_cdcr_name(facilities)

## Add Population and population attribute data

In [8]:
def merge_population_data(df, cdcr_pop):
    """
    Merges the 2025 population data into the facility GeoDataFrame.
    Uses 'cdcr_code' and 'Facility_Code' as the join keys.
    """

    cdcr_pop['Facility_Code'] = cdcr_pop['Facility_Code'].astype(str).str.strip()

    df = df.merge(
        cdcr_pop[['Facility_Code', 'Average_2025_Population']], 
        left_on='CDCR_code', 
        right_on='Facility_Code', 
        how='left'
    )

    if 'Facility_Code' in df.columns:
        df = df.drop(columns=['Facility_Code'])
    
    return df

In [9]:
facilities_pop = merge_population_data(facilities, cdcr_pop)

In [10]:
# Occupancy against CDCR design capacity: the 2025 mean of the monthly
# "Percent Occupied" column in the TPOP-1 reports (scrapers/extract_tpop1.py).
tpop = pd.read_csv("data/tpop1_institutions.csv")
tpop_2025 = tpop[tpop['report_date'].str.startswith('2025')]
occupancy = (
    (tpop_2025['pct_occupied'] / 100).groupby(tpop_2025['cdcr_code']).mean().round(4)
    .rename('capacity_percent_2025')
)
facilities_pop = facilities_pop.merge(occupancy, left_on='CDCR_code', right_index=True, how='left')

## Demographics

In [11]:
def clean_population_col(df):
    """
    Cleans the 'Population' column by:
    1. Removing commas.
    2. Converting '*', '**', and '(Blank)' to 0.
    3. Converting to numeric float.
    """
    df = df.copy()
    if 'Population' in df.columns:
        pop_series = df['Population'].astype(str).str.replace(',', '').str.strip()
        
        # Apply CDCR suppression rules and empty value logic
        # ** = < 50, using 25 as an estimate
        # * = < 10, using 5 as an estimate
        # (Blank) = 0
        pop_series = pop_series.replace('**', '25')
        pop_series = pop_series.replace('*', '5')
        pop_series = pop_series.replace(['(Blank)', 'nan', 'NaN', ''], '0')
        
        df['Population'] = pd.to_numeric(pop_series, errors='coerce').fillna(0)
    return df

In [12]:
def get_average_counts_df(age_df, cob_df, gender_df, race_df):
    """
    Produces a single DataFrame containing the average counts of all 
    variables across the four demographic topics, indexed by Location.
    """
    dfs = []
    
    # Process each topic into average counts per location and category
    for df, cat_col, prefix in [
        (age_df, 'Age Group', 'age'),
        (cob_df, 'Country of Birth', 'cob'),
        (gender_df, 'Gender', 'gender'),
        (race_df, 'Race', 'race')
    ]:
        temp = clean_population_col(df)
        # Average counts across time/months
        avg = temp.groupby(['Location', cat_col])['Population'].mean().reset_index()
        # Pivot to wide format
        pivot = avg.pivot(index='Location', columns=cat_col, values='Population').fillna(0)
        # Standardize column names
        pivot.columns = [f"{prefix}_{str(c).lower().replace(' ', '_').replace('/', '_').replace('-', '_')}" for c in pivot.columns]
        dfs.append(pivot)
        
    counts_df = pd.concat(dfs, axis=1).fillna(0)
    return counts_df

In [13]:
demographic_counts = get_average_counts_df(cdcr_age, cdcr_countryofbirth, cdcr_gender, cdcr_race)

In [14]:
def calculate_generic_percentages(facility_df, counts_df, prefix):
    """
    Calculates the average percentage of a demographic category per facility
    relative to the 'Average_2025_Population' column using pre-calculated counts.
    Used for Gender or other categories where no custom grouping is done.
    """
    category_cols = [c for c in counts_df.columns if c.startswith(f"{prefix}_")]
    pivot_df = counts_df[category_cols].copy()
    
    facility_df['CDCR_code'] = facility_df['CDCR_code'].astype(str).str.strip()
    pivot_df.index = pivot_df.index.astype(str).str.strip()
    
    merged_df = facility_df.merge(pivot_df, left_on='CDCR_code', right_index=True, how='left')
    
    for col in pivot_df.columns:
        pct_col_name = f"{col}_pct"
        ratios = np.where(
            merged_df['Average_2025_Population'] > 0,
            merged_df[col] / merged_df['Average_2025_Population'],
            0
        )
        # Round at .0001 and 1.0
        final_pacts = np.clip(ratios, 0, 1)
        merged_df[pct_col_name] = np.where(final_pacts < 0.001, 0, final_pacts)
            
    output_df = merged_df.drop(columns=list(pivot_df.columns))
    return output_df

In [15]:
def add_age_tier_demographics(df, counts_df):
    """
    Calculates age percentages using pre-calculated counts.

    Cumulative tiers (for health cost analysis):
      age_over_50_pct  — 50+
      age_over_55_pct  — 55+

    Decade brackets (to match CCHCS cost-per-age-bracket data):
      age_50_59_pct    — 50–59
      age_60_69_pct    — 60–69
      age_70_79_pct    — 70–79
      age_80plus_pct   — 80+
    """
    def age_col(name): return f"age_{name.lower().replace(' ', '_').replace('/', '_').replace('-', '_')}"

    groups_80_plus = [age_col(g) for g in ['80-84 Years', '85-89 Years', '90-94 Years', '95 and Older']]
    groups_70_79   = [age_col(g) for g in ['70-74 Years', '75-79 Years']]
    groups_60_69   = [age_col(g) for g in ['60-64 Years', '65-69 Years']]
    groups_50_59   = [age_col(g) for g in ['50-54 Years', '55-59 Years']]

    groups_55_plus = [age_col('55-59 Years')] + groups_60_69 + groups_70_79 + groups_80_plus
    groups_50_plus = [age_col('50-54 Years')] + groups_55_plus

    tiers = pd.DataFrame(index=counts_df.index)
    tiers['age_over_50_count'] = counts_df[counts_df.columns.intersection(groups_50_plus)].sum(axis=1)
    tiers['age_over_55_count'] = counts_df[counts_df.columns.intersection(groups_55_plus)].sum(axis=1)
    tiers['age_50_59_count']   = counts_df[counts_df.columns.intersection(groups_50_59)].sum(axis=1)
    tiers['age_60_69_count']   = counts_df[counts_df.columns.intersection(groups_60_69)].sum(axis=1)
    tiers['age_70_79_count']   = counts_df[counts_df.columns.intersection(groups_70_79)].sum(axis=1)
    tiers['age_80plus_count']  = counts_df[counts_df.columns.intersection(groups_80_plus)].sum(axis=1)

    df['CDCR_code'] = df['CDCR_code'].astype(str).str.strip()
    merged = df.merge(tiers, left_on='CDCR_code', right_index=True, how='left')

    count_cols = ['age_over_50_count', 'age_over_55_count',
                  'age_50_59_count', 'age_60_69_count', 'age_70_79_count', 'age_80plus_count']
    pct_cols   = ['age_over_50_pct',  'age_over_55_pct',
                  'age_50_59_pct',    'age_60_69_pct',  'age_70_79_pct',  'age_80plus_pct']

    for count_col, pct_col in zip(count_cols, pct_cols):
        if 'Average_2025_Population' in merged.columns:
            merged[pct_col] = np.where(
                merged['Average_2025_Population'] > 0,
                merged[count_col] / merged['Average_2025_Population'],
                0
            )
        else:
            merged[pct_col] = 0

    return merged.drop(columns=count_cols)


In [16]:
def calculate_race_percentages(facility_df, counts_df):
    """
    Specifically calculates percentages for race by aggregating into 
    white_pct and peopleofcolor_pct.
    """
    prefix = 'race'
    category_cols = [c for c in counts_df.columns if c.startswith(f"{prefix}_")]
    pivot_df = counts_df[category_cols].copy()
    
    facility_df['CDCR_code'] = facility_df['CDCR_code'].astype(str).str.strip()
    pivot_df.index = pivot_df.index.astype(str).str.strip()
    
    # Identify White vs POC
    white_col = f"{prefix}_white"
    other_cols = [c for c in category_cols if c != white_col]
    
    # Aggregate counts
    race_summary = pd.DataFrame(index=pivot_df.index)
    race_summary['white'] = pivot_df[white_col] if white_col in pivot_df.columns else 0
    race_summary['peopleofcolor'] = pivot_df[other_cols].sum(axis=1)

    merged_df = facility_df.merge(race_summary, left_on='CDCR_code', right_index=True, how='left')
    
    for col in ['white', 'peopleofcolor']:
        pct_col_name = f"race_{col}_pct"
        merged_df[pct_col_name] = np.where(
            merged_df['Average_2025_Population'] > 0,
            merged_df[col] / merged_df['Average_2025_Population'],
            0
        )
            
    output_df = merged_df.drop(columns=['white', 'peopleofcolor'])
    return output_df

In [17]:
f_pop_age = add_age_tier_demographics(facilities_pop, demographic_counts)

In [18]:
f_pop_gender = calculate_generic_percentages(f_pop_age, demographic_counts, 'gender')

In [19]:
f_pop_race = calculate_race_percentages(f_pop_gender, demographic_counts)

## Add manually reviewed misc facilities data

## Add housing cooling data (CDCR Air Cooling Pilot Supplemental Report, Jan 2026)

In [20]:
def merge_cooling_data(df, infra_path="sources/air_cooling_infrastructure_dec2025.csv"):
    """
    Merges per-facility housing cooling data from the CDCR Air Cooling Pilot
    Supplemental Report (Jan 2026, as of Dec 2025), at the housing-unit
    (wing/dorm/tier) level. This is the newest, complete, authoritative source.

    Produces:
      n_housing_units          - housing-unit count (denominator)
      pct_hu_mechanical  - fraction with mechanical (refrigerant) AC
      pct_hu_evaporative    - fraction with evaporative (swamp) cooling
      pct_hu_air_handlers    - fraction with air handlers only (no cooling)

    Mixed-cooling units are counted under each type they have, so the three shares
    can sum slightly > 1 (e.g. SATF evaporative = 1.03). The AC sub-indicator of the
    heat-risk index is 1 - pct_hu_mechanical; the app UI shows the three-way
    mix. See the cooling-data note in data/README.md.

    HISTORY: an older Reuters FOIA (2025) HVAC-equipment inventory previously produced
    a parallel `pct_units_*` / `n_housing_units` column set. It was incomplete (missed
    housing at 11 of 31 facilities — e.g. CIM read a spurious 100% refrigeration) and
    was dropped in 2026-07; only the CDCR report is used now.

    The report uses FSP and SQRC where cdcr_facilities uses FOL and SQ.
    """
    infra = pd.read_csv(infra_path)
    infra['institution'] = infra['institution'].replace({'FSP': 'FOL', 'SQRC': 'SQ'})
    n_hu = infra['n_housing_units']
    summary = pd.DataFrame({
        'cdcr_code_cooling': infra['institution'],
        'n_housing_units': n_hu,
        'pct_hu_evaporative':   (infra['hu_evaporative_cooling'] / n_hu).round(4),
        'pct_hu_mechanical': (infra['hu_mechanical_cooling']  / n_hu).round(4),
        'pct_hu_air_handlers':   (infra['hu_air_handlers_only']   / n_hu).round(4),
    })

    df = df.merge(summary, left_on='CDCR_code', right_on='cdcr_code_cooling', how='left')
    df = df.drop(columns=['cdcr_code_cooling'])

    return df

In [21]:
f_pop_cooling = merge_cooling_data(f_pop_race)

In [22]:
def merge_cdcr_metadata(gdf, cdcr_metadata):
    """
    Merges facility metadata (opening year, planned closure, etc.)
    """
    
    cdcr_metadata['Acronym'] = cdcr_metadata['Acronym'].astype(str).str.strip()

    gdf = gdf.merge(
        cdcr_metadata,
        left_on='CDCR_code',
        right_on='Acronym',
        how='left'
    )

    if 'Acronym' in gdf.columns:
        gdf = gdf.drop(columns=['Acronym'])

    # Lowercase new columns to maintain naming convention
    gdf.columns = [col.lower() for col in gdf.columns]

    return gdf

In [23]:
facilities_df = merge_cdcr_metadata(f_pop_cooling, cdcr_metadata)

## Add SB 601 Programs data

In [24]:
def merge_sb601_programs(df, sb601_path):
    """
    Adds two columns listing the programs each facility had operational capacity for
    during 2024-2025, based on the SB 601 dashboard data:
      - cognitive_behavioral_interventions: semicolon-separated list of CBI programs
      - rehabilitative_programs: semicolon-separated list of Rehabilitative Programs

    A program is included if its Operational Capacity was > 0 in any month of the year.
    Degree counts (e.g. Number of GED Certificates) are excluded because they have no
    Operational Capacity row.
    """
    sb601 = pd.read_csv(sb601_path, skiprows=1)  # skip the quarter-label row
    sb601.columns = ['institution', 'category', 'metric'] + list(sb601.columns[3:])

    # Keep only Operational Capacity rows
    capacity = sb601[sb601['metric'].str.endswith('- Operational Capacity')].copy()

    # Identify monthly columns before adding derived columns
    month_cols = capacity.columns[3:]  # Jul-24 through Apr-25

    # Convert monthly columns to numeric; treat blanks as 0
    for col in month_cols:
        capacity[col] = pd.to_numeric(capacity[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

    # Strip the " - Operational Capacity" suffix to get the program name
    capacity['program'] = capacity['metric'].str.replace(r'\s*-\s*Operational Capacity$', '', regex=True).str.strip()

    # A program "has capacity" if any monthly value > 0
    capacity = capacity[capacity[month_cols].max(axis=1) > 0]

    # Build semicolon-separated program lists per institution and category
    program_lists = (
        capacity.groupby(['institution', 'category'])['program']
        .apply(lambda progs: '; '.join(sorted(progs)))
        .reset_index()
    )

    col_map = {
        'Cognitive Behavioral Intervention': 'cognitive_behavioral_interventions',
        'Rehabilitative Programs': 'rehabilitative_programs',
    }

    for category_label, col_name in col_map.items():
        subset = program_lists[program_lists['category'] == category_label][['institution', 'program']]
        subset = subset.rename(columns={'institution': 'cdcr_code', 'program': col_name})
        df = df.merge(subset, on='cdcr_code', how='left')

    return df

In [25]:
facilities_df = merge_sb601_programs(
    facilities_df,
    "data/sb601_programs.csv"
)

## Add CCHCS Health Care Population data (2025 averages)

In [26]:
def merge_cchcs_ipc(df, ipc_path):
    """
    Adds 2025 average CCHCS health classification percentages per facility.

    Source: CCHCS Health Care Services Dashboard, cchcs.ca.gov/dashboard/
    Measures are expressed as % of total facility population (confirmed: the four
    risk tiers sum to 100% for every facility-month, meaning the denominator is
    the total endorsed population, not a subset of active patients).

    Columns added:
      cchcs_high_risk_p1_pct_2025   - % High Risk Priority 1
      cchcs_high_risk_p2_pct_2025   - % High Risk Priority 2
      cchcs_medium_risk_pct_2025    - % Medium Risk
      cchcs_low_risk_pct_2025       - % Low Risk
      cchcs_mental_health_eop_pct_2025       - % Mental Health EOP
      cchcs_dpp_pct_2025            - % Disability Placement Program (DPP) Patients
      cchcs_age_over_50_pct_2025    - % Patients 50 Years or Older
      cchcs_specialized_beds_2025   - Specialized Health Care Beds (count, 12-month avg)
    """
    ipc = pd.read_csv(ipc_path)
    ipc['val'] = pd.to_numeric(
        ipc['value'].astype(str).str.replace('%', '').str.strip(), errors='coerce'
    )

    # Remap CCHCS codes that differ from our cdcr_code convention
    ipc['institution'] = ipc['institution'].replace({'FSP': 'FOL'})

    measures = {
        'High Risk Priority 1':                        'cchcs_high_risk_p1_pct_2025',
        'High Risk Priority 2':                        'cchcs_high_risk_p2_pct_2025',
        'Medium Risk':                                 'cchcs_medium_risk_pct_2025',
        'Low Risk':                                    'cchcs_low_risk_pct_2025',
        'Mental Health EOP':                           'cchcs_mental_health_eop_pct_2025',
        'Disability Placement Program (DPP) Patients': 'cchcs_dpp_pct_2025',
        'Patients 50 Years or Older':                  'cchcs_age_over_50_pct_2025',
        'Specialized Health Care Beds':                'cchcs_specialized_beds_2025',
    }

    ipc_2025 = ipc[ipc['month'].str.endswith('2025') & ipc['measure'].isin(measures)]

    for measure, col_name in measures.items():
        avg = (
            ipc_2025[ipc_2025['measure'] == measure]
            .groupby('institution')['val']
            .mean()
            .round(1)
            .reset_index()
            .rename(columns={'institution': 'cdcr_code', 'val': col_name})
        )
        df = df.merge(avg, on='cdcr_code', how='left')

    return df


In [27]:
facilities_df = merge_cchcs_ipc(
    facilities_df,
    "data/cchcs_ipc.csv"
)

## Add SCO Staffing data (2025 averages)

In [28]:
def merge_sco_staffing(df, staffing_path):
    """
    Adds 2025 average SCO staff headcount per facility.

    sco_state_staff_2025:       CDCR operational + CCHCS healthcare employees
                                (non-PIA rows). For CHCF, the CCHCS row is summed
                                with the CDCR operational row.
    sco_incarcerated_staff_2025: Prison Industry Authority workers (PIA rows),
                                who are incarcerated people employed through PIA.
    """
    sco = pd.read_csv(staffing_path)

    state = (
        sco[~sco['is_pia']].groupby('cdcr_code')['total']
        .sum().round(1).reset_index()
        .rename(columns={'total': 'sco_state_staff_2025'})
    )
    incarcerated = (
        sco[sco['is_pia']].groupby('cdcr_code')['total']
        .sum().round(1).reset_index()
        .rename(columns={'total': 'sco_incarcerated_staff_2025'})
    )
    df = df.merge(state, on='cdcr_code', how='left')
    df = df.merge(incarcerated, on='cdcr_code', how='left')
    return df


In [29]:
facilities_df = merge_sco_staffing(
    facilities_df,
    "data/sco_staffing_avg.csv"
)


## Add restricted housing data (2025 average)

In [30]:
def merge_restricted_housing(df, rh_path):
    """
    Adds rhu_pct_2025: the mean of the monthly % of population in restricted
    housing units across the twelve 2025 STA429 reports.
    """
    rh = pd.read_csv(rh_path)
    rh_2025 = rh[rh['data_month'].str.startswith('2025')]
    avg = (
        rh_2025.groupby('cdcr_code')['pct_in_rhu'].mean().round(4)
        .reset_index()
        .rename(columns={'pct_in_rhu': 'rhu_pct_2025'})
    )
    return df.merge(avg, on='cdcr_code', how='left')

In [31]:
facilities_df = merge_restricted_housing(
    facilities_df,
    "data/restricted_housing.csv"
)

## Add violent incidents (annual average)

In [32]:
def merge_violent_incidents(df, incidents_path):
    """
    Adds avg_annual_violent_incidents: total violent incidents divided by the
    number of calendar years with any reported months (2021-2025). Facilities
    that closed during the period (CAC, CVSP) are averaged over their open years.
    """
    vi = pd.read_csv(incidents_path)
    by_fac = vi.groupby('facility').agg(
        total=('violent_incidents', 'sum'), n_years=('year', 'nunique')
    )
    avg = (by_fac['total'] / by_fac['n_years']).round(1).rename('avg_annual_violent_incidents')
    return df.merge(avg, left_on='cdcr_code', right_index=True, how='left')

In [33]:
facilities_df = merge_violent_incidents(
    facilities_df,
    "data/cdcr_violent_incidents_by_facility.csv"
)

## Add elevation (USGS EPQS)

In [34]:
# Ground elevation at the facility centroid, from scrapers/fetch_facility_elevation.py
elevation = pd.read_csv("../ca_prison_climate_justice/data_sources/facilities/facility_elevation.csv")
facilities_df = facilities_df.merge(
    elevation[['facilityid', 'elevation_m']].assign(elevation_m=lambda d: d['elevation_m'].round(0)),
    on='facilityid', how='left'
)

## Column order

In [35]:
# uhi_normalized (Benz & Burney surface UHI) passes through from ca_facilities.csv.
OUTPUT_COLUMNS = [
    'facilityid', 'name', 'address', 'city', 'state', 'zip', 'telephone', 'type', 'status',
    'population', 'county', 'countyfips', 'country', 'website', 'securelvl', 'capacity',
    'geometry', 'longitude', 'latitude', 'tract_geoid', 'capacity_percent',
    'dist_nearest_medical_mi', 'in_urban_area_2020', 'cdcr_code', 'cdcr_firecamp',
    'average_2025_population', 'age_over_50_pct', 'age_over_55_pct', 'age_50_59_pct',
    'age_60_69_pct', 'age_70_79_pct', 'age_80plus_pct', 'gender_female_pct',
    'gender_male_pct', 'race_white_pct', 'race_peopleofcolor_pct', 'n_housing_units',
    'pct_hu_evaporative', 'pct_hu_mechanical', 'pct_hu_air_handlers', 'year_opened',
    'year_opened_notes', 'planned_closure', 'california_model_facility',
    'cdcr_air_cooling_pilot', 'cognitive_behavioral_interventions',
    'rehabilitative_programs', 'cchcs_high_risk_p1_pct_2025',
    'cchcs_high_risk_p2_pct_2025', 'cchcs_medium_risk_pct_2025', 'cchcs_low_risk_pct_2025',
    'cchcs_mental_health_eop_pct_2025', 'cchcs_dpp_pct_2025', 'cchcs_age_over_50_pct_2025',
    'cchcs_specialized_beds_2025', 'sco_state_staff_2025', 'sco_incarcerated_staff_2025',
    'uhi_normalized', 'elevation_m', 'avg_annual_violent_incidents', 'rhu_pct_2025',
    'capacity_percent_2025',
]
facilities_df = facilities_df[OUTPUT_COLUMNS]

# Site attributes are kept for CDCR institutions only (not fire camps)
facilities_df['cdcr_code'] = facilities_df['cdcr_code'].replace('None', np.nan)
not_cdcr = facilities_df['cdcr_code'].isna()
facilities_df.loc[not_cdcr, ['uhi_normalized', 'elevation_m']] = np.nan

## Export

In [36]:
facilities_df.to_csv("data/cdcr_facilities.csv", index=False)